# 📊 Customer Churn Analysis — Phase 3: EDA & Feature Engineering

---

## What this notebook does

We take `master_clean.csv` from Phase 2 and do two things in strict order:

1. **Exploratory Data Analysis (EDA)** — understand what the data looks like before touching it. Distributions, churn rates by segment, statistical tests, correlations. EDA tells us *what to engineer*.
2. **Feature Engineering** — transform raw columns into model-ready features. Encode categoricals, create new composite signals, scale numerics, and produce a final feature matrix `X` and target vector `y`.

## What we inherit from Phase 2

| Column group | Columns | State |
|---|---|---|
| Customer profile | `gender`, `senior_citizen`, `partner`, `dependents`, `tenure`, `phone_service`, `multiple_lines`, `internet_service`, `online_security`, `online_backup`, `device_protection`, `tech_support`, `streaming_tv`, `streaming_movies`, `contract`, `paperless_billing`, `payment_method`, `monthly_charges`, `total_charges` | Cleaned, snake_case, no nulls |
| Missing flags | `tenure_was_missing`, `monthly_charges_was_missing` | Binary 0/1 |
| Usage behaviour | `total_logins`, `avg_session_mins`, `login_days_active`, `days_since_last_login` | Aggregated, 0-filled |
| Support tickets | `total_tickets`, `avg_resolution_hours`, `max_resolution_hours`, `billing_tickets`, `technical_tickets`, `billing_ticket_ratio` | Aggregated, 0-filled |
| Billing behaviour | `total_bills`, `total_late_payments`, `late_payment_rate`, `avg_payment_gap`, `max_payment_gap`, `avg_amount_due` | Aggregated, 0-filled |
| Target | `churn` | Binary int 0/1, ~26% positive rate |

## Outputs produced

| File | Contents |
|---|---|
| `X_raw.csv` | Unscaled feature matrix (for tree-based models) |
| `X_scaled.csv` | StandardScaler-normalised features (for linear/distance models) |
| `y.csv` | Binary target vector |
| `scaler.pkl` | Fitted StandardScaler (refit on train-only in notebook 04) |
| `feature_names.txt` | Ordered list of all feature column names |

## Notebook Map
```
01_data_collection.ipynb
02_cleaning_merging.ipynb
03_eda_feature_engineering.ipynb   <- YOU ARE HERE
04_modelling.ipynb
```

---
## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import mannwhitneyu, chi2_contingency
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

BASE_PATH      = "/content/drive/MyDrive/customer-churn-project"
PROCESSED_PATH = os.path.join(BASE_PATH, "data/processed/")

# ── Plot style ────────────────────────────────────────────────────────
# whitegrid removes chartjunk while keeping light reference lines.
# Consistent palette declared once — every plot reuses C0/C1 so a reviewer
# can scan the whole notebook and blue always means no-churn, red always churned.
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi":        110,
    "axes.titlesize":    12,
    "axes.titleweight":  "bold",
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

C0 = "#4C9BE8"   # blue  = did not churn
C1 = "#E8654C"   # red   = churned
# Both int and string keys prevent KeyError regardless of pandas returning int64 or plain int
CHURN_PALETTE = {0: C0, 1: C1, "0": C0, "1": C1}

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

print("Setup complete.")
print(f"pandas {pd.__version__} | numpy {np.__version__}")

---
## 2. Load Master Table & Integrity Check

We load `master_clean.csv` from Phase 2 and immediately run a hard validation:
- Zero nulls (confirmed clean by notebook 02)
- Behavioural columns have non-zero values (confirms the customer_id join worked)

If the behavioural check fails, the merge in notebook 02 silently failed — re-run notebook 02 before continuing.

In [ ]:
df = pd.read_csv(f"{PROCESSED_PATH}master_clean.csv")

print(f"Shape:      {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Churn rate: {df['churn'].mean():.1%}  (expected ~26%)")

nulls = df.isna().sum().sum()
print(f"Total nulls: {nulls} {'ok' if nulls == 0 else '<- WARNING rerun notebook 02'}")

# Behavioural sanity check — if any show non-zero=0 the merge failed
behav_check = [
    "total_logins", "avg_session_mins", "total_tickets",
    "avg_payment_gap", "late_payment_rate", "days_since_last_login"
]
print("\nBehavioural column sanity (non-zero row counts):")
all_ok = True
for col in behav_check:
    if col not in df.columns:
        print(f"  {col:<30} MISSING FROM DATAFRAME")
        all_ok = False
        continue
    nz     = (df[col] != 0).sum()
    status = "ok" if nz > 0 else "<- WARNING all zeros - rerun notebook 02"
    print(f"  {col:<30} non-zero={nz:>5,}  {status}")
    if nz == 0:
        all_ok = False

if not all_ok:
    raise ValueError(
        "Behavioural columns are all-zero. "
        "The customer_id join in notebook 02 failed. Re-run notebook 02."
    )

print("\nAll checks passed ok")
print(f"\nAll {df.shape[1]} columns:")
print(df.columns.tolist())

---
## 3. Column Catalogue

We explicitly categorise every column before touching it. This catalogue controls:
- Which EDA plot type gets applied (KDE/box for numeric, bar for categorical)
- Which statistical test gets used (Mann-Whitney for numeric, Chi-squared for categorical)
- Which encoding strategy gets applied (ordinal, one-hot, binary map)
- Which columns get scaled (continuous/count only, not binary/ordinal)

In [ ]:
TARGET = "churn"

DROP_COLS = [
    "customer_id",                  # identifier
    "total_charges",                # = tenure x monthly_charges -> multicollinear
    "tenure_was_missing",           # internal cleaning flag
    "monthly_charges_was_missing",  # internal cleaning flag
]

BINARY_COLS = [
    "senior_citizen", "partner", "dependents",
    "phone_service", "paperless_billing"
]

CATEGORICAL_COLS = [
    "gender", "multiple_lines", "internet_service",
    "online_security", "online_backup", "device_protection",
    "tech_support", "streaming_tv", "streaming_movies",
    "contract", "payment_method"
]

CONTINUOUS_COLS = [
    "tenure", "monthly_charges",
    "avg_session_mins", "avg_resolution_hours", "max_resolution_hours",
    "avg_payment_gap", "max_payment_gap", "avg_amount_due"
]

COUNT_COLS = [
    "total_logins", "login_days_active", "days_since_last_login",
    "total_tickets", "billing_tickets", "technical_tickets",
    "total_bills", "total_late_payments"
]

RATE_COLS    = ["late_payment_rate", "billing_ticket_ratio"]
NUMERIC_COLS = CONTINUOUS_COLS + COUNT_COLS + RATE_COLS

# Verify catalogue covers the full dataframe
catalogued       = set(BINARY_COLS + CATEGORICAL_COLS + NUMERIC_COLS + DROP_COLS + [TARGET])
uncatalogued     = set(df.columns) - catalogued
missing_from_df  = (catalogued - set(df.columns)) - set(DROP_COLS) - {TARGET}

print(f"Target:      {TARGET}")
print(f"Drop:        {len(DROP_COLS)} cols")
print(f"Binary:      {len(BINARY_COLS)} cols")
print(f"Categorical: {len(CATEGORICAL_COLS)} cols")
print(f"Numeric:     {len(NUMERIC_COLS)} cols")
print(f"Uncatalogued (will need attention): {uncatalogued if uncatalogued else 'none ok'}")
print(f"In catalogue but missing from df:   {missing_from_df if missing_from_df else 'none ok'}")

---
## 4. EDA — Target Variable

### 4a. Class distribution

**Why look at class balance first?** Because it determines which metrics are valid. A model that always predicts no-churn achieves ~74% accuracy on this dataset but identifies zero actual churners — useless for the business. Understanding the imbalance upfront forces us to choose appropriate metrics (ROC-AUC, PR-AUC) and appropriate modelling strategies (class weighting, oversampling) before we touch any model.

In [ ]:
churn_counts = df[TARGET].value_counts().sort_index()
churn_pct    = df[TARGET].value_counts(normalize=True).sort_index() * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

bars = axes[0].bar(
    ["Did not churn", "Churned"],
    churn_counts.values,
    color=[C0, C1], width=0.5, edgecolor="white", linewidth=1.5
)
for bar, pct in zip(bars, churn_pct.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 40,
        f"{pct:.1f}%", ha="center", va="bottom", fontsize=10
    )
axes[0].set_title("Churn class distribution")
axes[0].set_ylabel("Customers")
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{int(x):,}")
)

axes[1].pie(
    churn_counts.values,
    labels=["Did not churn", "Churned"],
    colors=[C0, C1], autopct="%1.1f%%", startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2}
)
axes[1].set_title("Churn proportion")

plt.suptitle("Target variable: Churn", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print(f"Did not churn: {churn_counts[0]:,}  ({churn_pct[0]:.1f}%)")
print(f"Churned:       {churn_counts[1]:,}  ({churn_pct[1]:.1f}%)")
print(f"Imbalance ratio: {churn_counts[0] / churn_counts[1]:.1f}:1")
print()
print("Modelling implications:")
print("  - Accuracy is misleading -- 74% achievable by always predicting no-churn")
print("  - Primary metrics: ROC-AUC and PR-AUC")
print("  - Handle imbalance: class_weight=balanced and/or SMOTE in notebook 04")

---
## 5. EDA — Continuous Features vs Churn

### 5a. KDE distributions split by churn

**Why KDE instead of histograms?** A histogram's shape depends on the arbitrary bin-width choice. KDE gives the true underlying distribution shape. When churned and not-churned KDE curves are overlaid, a horizontal shift immediately tells you the feature is predictive — no calculation needed.

**Why `.values` on the rug sample?** Passing a pandas Series to `ax.plot()` makes matplotlib use the Series index as x-coordinates, not the values, producing wrong rug positions. `.values` strips the index and passes a plain numpy array, which is the correct input.

In [ ]:
kde_cols = [c for c in [
    "tenure", "monthly_charges",
    "avg_session_mins", "days_since_last_login",
    "avg_payment_gap", "late_payment_rate",
    "avg_resolution_hours", "total_logins"
] if c in df.columns]

n_cols = 4
n_rows = int(np.ceil(len(kde_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(kde_cols):
    ax = axes[i]
    for churn_val, label in [(0, "Did not churn"), (1, "Churned")]:
        subset = df[df[TARGET].astype(int) == churn_val][col].dropna()
        color  = CHURN_PALETTE[churn_val]

        sns.kdeplot(
            subset, ax=ax, label=label,
            color=color, fill=True, alpha=0.35, linewidth=1.8
        )

        # Rug marks: .values strips the pandas index before passing to matplotlib
        rug = subset.sample(min(300, len(subset)), random_state=42).values
        ax.plot(
            rug,
            np.full(len(rug), ax.get_ylim()[0]),
            "|", color=color, alpha=0.25, markersize=4
        )

    ax.set_title(col.replace("_", " ").title(), fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

for j in range(len(kde_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Continuous feature distributions by churn status",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 5b. Box plots — median and spread by churn

**Why box plots in addition to KDE?** KDE shows shape but hides exact summary statistics. Box plots give Q1, median, Q3, whiskers, and individual outliers explicitly. Together they give a complete picture: KDE for shape, box plots for the numbers.

**Why pure matplotlib boxplot and not seaborn?** Seaborn 0.13 has a known `UnboundLocalError: cannot access local variable 'boxprops'` bug that triggers when certain palette configurations are passed. Pure matplotlib `axes.boxplot()` is what seaborn wraps internally anyway — we cut out the broken wrapper and colour the patches directly afterward.

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(kde_cols):
    ax    = axes[i]
    g0    = df[df[TARGET].astype(int) == 0][col].dropna().values
    g1    = df[df[TARGET].astype(int) == 1][col].dropna().values
    med0  = float(np.median(g0)) if len(g0) > 0 else float("nan")
    med1  = float(np.median(g1)) if len(g1) > 0 else float("nan")

    bp = ax.boxplot(
        [g0, g1],
        positions=[0, 1], widths=0.45,
        patch_artist=True,
        medianprops ={"color": "white",  "linewidth": 2.0},
        whiskerprops={"linewidth": 1.2},
        capprops    ={"linewidth": 1.2},
        flierprops  ={"marker": "o", "markersize": 2.5,
                      "alpha": 0.35, "linestyle": "none"}
    )
    bp["boxes"][0].set_facecolor(C0); bp["boxes"][0].set_alpha(0.75)
    bp["boxes"][1].set_facecolor(C1); bp["boxes"][1].set_alpha(0.75)

    ax.set_title(
        f"{col.replace('_', ' ').title()}\n"
        f"median: no={med0:.1f}, yes={med1:.1f}",
        fontsize=9
    )
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["No churn", "Churned"], fontsize=9)

for j in range(len(kde_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Box plots of continuous features by churn status",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 5c. Mann-Whitney U test — statistical significance and effect size

**Why Mann-Whitney instead of a t-test?** The t-test requires both groups to be normally distributed. From the KDE plots above, most numeric columns are right-skewed (total_logins, resolution_time_hours, payment_gap all have long right tails). Mann-Whitney makes no distributional assumption and is the correct non-parametric alternative.

**Why report rank-biserial r alongside the p-value?** With ~7,000 rows, almost every difference will be significant (p < 0.05). The p-value only tells you a difference exists. Rank-biserial r tells you how *large* it is — values above 0.3 are medium effect, above 0.5 are large. We use this to rank features before engineering.

In [ ]:
mwu_results = []
for col in NUMERIC_COLS:
    if col not in df.columns:
        continue
    g0 = df[df[TARGET].astype(int) == 0][col].dropna()
    g1 = df[df[TARGET].astype(int) == 1][col].dropna()
    if len(g0) < 2 or len(g1) < 2:
        continue

    stat, p = mannwhitneyu(g0, g1, alternative="two-sided")
    r       = 1 - (2 * stat) / (len(g0) * len(g1))   # rank-biserial correlation

    mwu_results.append({
        "feature":         col,
        "median_no_churn": round(g0.median(), 3),
        "median_churned":  round(g1.median(), 3),
        "p_value":         round(p, 6),
        "effect_size_r":   round(abs(r), 3),
        "significant":     "yes" if p < 0.05 else "no"
    })

mwu_df = pd.DataFrame(mwu_results).sort_values("effect_size_r", ascending=False)
print("Mann-Whitney U results sorted by effect size:")
print(mwu_df.to_string(index=False))

---
## 6. EDA — Categorical Features vs Churn

### 6a. Churn rate by category

**Why churn rate per category instead of raw counts?** A large category always has more churners in raw count terms even if its *rate* is identical to a small one. Rates normalise by group size and give a fair comparison. The dashed overall churn rate line lets you immediately see which categories are above or below average risk.

In [ ]:
overall_rate = df[TARGET].mean()

n_cols_plot = 3
n_rows_plot = int(np.ceil(len(CATEGORICAL_COLS) / n_cols_plot))
fig, axes   = plt.subplots(n_rows_plot, n_cols_plot,
                            figsize=(18, n_rows_plot * 4.5))
axes = axes.flatten()

for i, col in enumerate(CATEGORICAL_COLS):
    if col not in df.columns:
        continue
    ax = axes[i]

    summary = (
        df.groupby(col)[TARGET]
        .agg(churn_rate="mean", n="count")
        .reset_index()
        .sort_values("churn_rate", ascending=False)
    )

    bar_colors = [C1 if r > overall_rate else C0 for r in summary["churn_rate"]]
    bars = ax.bar(
        summary[col], summary["churn_rate"],
        color=bar_colors, edgecolor="white", linewidth=1.2, width=0.6
    )
    for bar, (_, row) in zip(bars, summary.iterrows()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{row['churn_rate']:.0%}\nn={row['n']:,}",
            ha="center", va="bottom", fontsize=7.5, linespacing=1.4
        )

    ax.axhline(overall_rate, color="grey", linestyle="--",
               linewidth=1, label=f"Avg {overall_rate:.0%}")
    ax.set_title(col.replace("_", " ").title(), fontsize=10)
    ax.set_ylabel("Churn rate")
    ax.set_ylim(0, min(1.0, summary["churn_rate"].max() * 1.3))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.tick_params(axis="x", rotation=25, labelsize=8)
    ax.legend(fontsize=8)

for j in range(len(CATEGORICAL_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Churn rate by categorical feature",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 6b. Chi-squared test with Cramér's V

**Why Chi-squared for categoricals?** Mann-Whitney requires numbers. Chi-squared tests whether the churn distribution across category levels differs from chance.

**Why Cramér's V?** Chi-squared grows with sample size. With 7,000 rows everything is significant. Cramér's V normalises to 0-1 regardless of n or number of categories. Values above 0.1 are practically meaningful.

In [ ]:
chi2_results = []
for col in CATEGORICAL_COLS + BINARY_COLS:
    if col not in df.columns:
        continue
    ct              = pd.crosstab(df[col], df[TARGET])
    chi2, p, dof, _ = chi2_contingency(ct)
    n               = ct.sum().sum()
    min_dim         = min(ct.shape) - 1
    cramers         = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0

    chi2_results.append({
        "feature":     col,
        "chi2":        round(chi2, 2),
        "p_value":     round(p, 6),
        "cramers_v":   round(cramers, 4),
        "significant": "yes" if p < 0.05 else "no"
    })

chi2_df = pd.DataFrame(chi2_results).sort_values("cramers_v", ascending=False)
print("Chi-squared results sorted by Cramer's V:")
print(chi2_df.to_string(index=False))

---
## 7. EDA — Correlation Analysis

### 7a. Spearman correlation heatmap

**Why Spearman instead of Pearson?** Pearson assumes linear relationships and is heavily influenced by outliers. Spearman uses ranks — it measures whether Y tends to increase as X increases regardless of linearity, and is robust to the skewed distributions we saw in Section 5.

We flag pairs with |r| > 0.75 as candidates for feature removal before modelling.

In [ ]:
corr_cols   = [c for c in NUMERIC_COLS + [TARGET] if c in df.columns]
corr_matrix = df[corr_cols].corr(method="spearman")
mask        = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(
    corr_matrix, mask=mask, ax=ax,
    cmap="RdBu_r", vmin=-1, vmax=1,
    annot=True, fmt=".2f", annot_kws={"size": 7},
    linewidths=0.3, linecolor="white",
    cbar_kws={"label": "Spearman r", "shrink": 0.7}
)
ax.set_title("Spearman correlation matrix — numeric features + churn",
             fontsize=12, fontweight="bold", pad=10)
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0,  labelsize=8)
plt.tight_layout()
plt.show()

print("Highly correlated feature pairs (|r| > 0.75):")
found = False
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.75:
            print(f"  {corr_matrix.columns[i]:<28} <-> {corr_matrix.columns[j]:<28}  r = {r:.3f}")
            found = True
if not found:
    print("  None above 0.75 ok")

### 7b. Feature correlation with churn — ranked bar chart

In [ ]:
churn_corr = (
    corr_matrix[TARGET].drop(TARGET)
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 7))
colors  = [C1 if v > 0 else C0 for v in churn_corr.values]
bars    = ax.barh(
    churn_corr.index, churn_corr.values,
    color=colors, edgecolor="white", linewidth=0.8
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Spearman correlation with churn")
ax.set_title("Numeric feature correlation with churn", fontweight="bold")

for bar, val in zip(bars, churn_corr.values):
    ax.text(
        val + (0.004 if val >= 0 else -0.004),
        bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}", va="center",
        ha="left" if val >= 0 else "right", fontsize=8
    )
plt.tight_layout()
plt.show()

---
## 8. EDA — Key Business Insights

### 8a. Churn rate by contract x tenure band

A pivot heatmap shows the *interaction* between two variables — whether the effect of contract type on churn changes with tenure. A simple bar chart would show one at a time. This is one of the most actionable views in the notebook: new month-to-month customers are the highest priority intervention group.

In [ ]:
df["tenure_band"] = pd.cut(
    df["tenure"],
    bins=[0, 12, 36, 60, 73],
    labels=["New (0-12m)", "Developing (13-36m)",
             "Established (37-60m)", "Loyal (61-72m)"],
    include_lowest=True
)

pivot   = df.pivot_table(values=TARGET, index="contract",
                          columns="tenure_band", aggfunc="mean")
pivot_n = df.pivot_table(values=TARGET, index="contract",
                          columns="tenure_band", aggfunc="count")

fig, ax = plt.subplots(figsize=(11, 3.5))
sns.heatmap(
    pivot, ax=ax, annot=True, fmt=".0%",
    cmap="YlOrRd", vmin=0, vmax=0.65,
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Churn rate",
              "format": mticker.PercentFormatter(1.0)}
)
ax.set_title("Churn rate by contract type x tenure band",
             fontweight="bold", pad=10)
ax.set_xlabel("Tenure band")
ax.set_ylabel("Contract type")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

print("Sample sizes per cell:")
print(pivot_n.to_string())

### 8b. Monthly charges by contract x churn

**Why violin plots?** Two categoricals (contract, churn) + one continuous (monthly_charges). A violin shows the full distribution shape *and* the IQR summary simultaneously. `split=True` draws churned/not-churned as the two halves of the same violin — more space-efficient than side-by-side and easier to compare.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Map churn to a readable label column to avoid palette dict bug
plot_df = df.copy()
plot_df["churn_label"] = plot_df[TARGET].map({0: "No churn", 1: "Churned"})

sns.violinplot(
    data=plot_df,
    x="contract", y="monthly_charges",
    hue="churn_label",
    hue_order=["No churn", "Churned"],
    palette={"No churn": C0, "Churned": C1},
    split=True, inner="quartile",
    linewidth=1.2, ax=ax
)
ax.set_title("Monthly charges: contract type x churn", fontweight="bold")
ax.set_xlabel("Contract type")
ax.set_ylabel("Monthly charges ($)")
ax.legend(title="Churn", fontsize=9)
plt.tight_layout()
plt.show()

### 8c. Behavioural signals by churn

In [ ]:
behav_plot = [
    ("days_since_last_login",  "Days since last login"),
    ("total_logins",           "Total logins"),
    ("late_payment_rate",      "Late payment rate"),
    ("avg_payment_gap",        "Avg payment gap ($)"),
    ("total_tickets",          "Total support tickets"),
    ("billing_ticket_ratio",   "Billing ticket ratio"),
]
behav_plot = [(c, l) for c, l in behav_plot if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, (col, label) in enumerate(behav_plot):
    ax = axes[i]
    g0 = df[df[TARGET].astype(int) == 0][col].dropna().values
    g1 = df[df[TARGET].astype(int) == 1][col].dropna().values

    bp = ax.boxplot(
        [g0, g1], positions=[0, 1], widths=0.45,
        patch_artist=True,
        medianprops ={"color": "white", "linewidth": 2},
        whiskerprops={"linewidth": 1.2},
        capprops    ={"linewidth": 1.2},
        flierprops  ={"marker": "o", "markersize": 2,
                      "alpha": 0.3, "linestyle": "none"}
    )
    bp["boxes"][0].set_facecolor(C0); bp["boxes"][0].set_alpha(0.75)
    bp["boxes"][1].set_facecolor(C1); bp["boxes"][1].set_alpha(0.75)

    ax.set_title(label, fontweight="bold", fontsize=10)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["No churn", "Churned"], fontsize=9)

for j in range(len(behav_plot), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Behavioural signals by churn status",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 9. Feature Engineering

EDA is complete. We now transform raw columns into a model-ready feature matrix.

### 9a. New composite features

Each feature below addresses a specific pattern identified in the EDA. We document the business motivation for every one.

In [ ]:
fe = df.copy()

# 1. avg_monthly_spend: total_charges / tenure
#    Normalises cumulative spend by how long the customer has been active.
#    A customer who paid 5k over 6 years is very different from one who paid
#    5k in 6 months. The raw totals look the same; this ratio does not.
fe["avg_monthly_spend"] = np.where(
    fe["tenure"] > 0,
    fe["total_charges"] / fe["tenure"],
    fe["monthly_charges"]   # for tenure=0 customers, use monthly as proxy
)

# 2. charge_vs_tier_avg: monthly_charges minus the mean for their internet tier.
#    Unusually high charges relative to peers signal billing frustration -> churn.
tier_mean                = fe.groupby("internet_service")["monthly_charges"].transform("mean")
fe["charge_vs_tier_avg"] = (fe["monthly_charges"] - tier_mean).round(2)

# 3. logins_per_active_day: total_logins / login_days_active.
#    Separates habitual daily users from binge-then-disappear patterns.
#    A customer logging in every day is much stickier than one binging for
#    two days and then going quiet for months.
fe["logins_per_active_day"] = np.where(
    fe["login_days_active"] > 0,
    (fe["total_logins"] / fe["login_days_active"]).round(3),
    0
)

# 4. num_services: count of add-on services subscribed.
#    More services = higher switching cost = lower churn risk (stickiness).
#    These columns are still Title Case strings at this point.
svc_cols          = ["online_security", "online_backup", "device_protection",
                     "tech_support", "streaming_tv", "streaming_movies"]
fe["num_services"] = sum(
    (fe[c].astype(str).str.strip().str.title() == "Yes").astype(int)
    for c in svc_cols if c in fe.columns
)

# 5. risk_score: additive count of the 4 strongest individual churn predictors.
#    Human-interpretable for business reporting ('this customer has 3 of 4 risk factors').
#    Also a model feature capturing compound risk that linear models miss.
fe["risk_score"] = (
    (fe["contract"].astype(str).str.strip().str.title() == "Month-To-Month").astype(int) +
    (fe["tenure"]            < 12).astype(int) +
    (fe["late_payment_rate"] >  0).astype(int) +
    (fe["total_tickets"]     >  2).astype(int)
)

new_feats = ["avg_monthly_spend", "charge_vs_tier_avg",
             "logins_per_active_day", "num_services", "risk_score"]
print("New composite features:")
print(fe[new_feats].describe().T.to_string())

nulls_new = fe[new_feats].isna().sum()
print(f"\nNulls introduced: {nulls_new[nulls_new>0].to_dict() if nulls_new.any() else 'none ok'}")

### 9b. Encode binary text columns (Yes/No -> 1/0)

**Why explicit `.map()` instead of `LabelEncoder`?** `LabelEncoder` assigns labels alphabetically. For `No`/`Yes` that happens to be 0/1, which is correct, but for other string pairs the order is arbitrary and potentially misleading. `.map()` is explicit and self-documenting.

**Why map `'No Internet Service'` and `'No Phone Service'` to 0?** These appear in service add-on columns when the base service is not subscribed. They are semantically equivalent to `No` — the customer definitely does not have the add-on.

In [ ]:
yes_no_cols = [
    "partner", "dependents", "phone_service", "paperless_billing",
    "multiple_lines",
    "online_security", "online_backup", "device_protection",
    "tech_support", "streaming_tv", "streaming_movies"
]

yn_map = {
    "Yes": 1, "No": 0,
    "No Internet Service": 0,
    "No Phone Service":    0
}

for col in yes_no_cols:
    if col not in fe.columns:
        continue
    fe[col] = (
        fe[col].astype(str).str.strip().str.title()
        .map(yn_map)
        .fillna(0)
        .astype(int)
    )

if "gender" in fe.columns:
    fe["gender"] = (
        fe["gender"].astype(str).str.strip().str.title()
        .map({"Male": 1, "Female": 0})
        .fillna(0).astype(int)
    )

print("Binary encoding complete.")
for col in ["partner", "phone_service", "online_security"]:
    if col in fe.columns:
        print(f"  {col}: {fe[col].value_counts().to_dict()}")

### 9c. Encode multi-class categorical columns

**Why ordinal for `contract`?** Month-to-Month, One Year, Two Year have a natural order — increasing commitment level. Encoding as 0, 1, 2 preserves this. One-hot would treat them as equally unrelated categories and lose the ordering signal.

**Why one-hot for `internet_service` and `payment_method`?** No natural order. `Fiber Optic` is not greater than `Dsl`. One-hot makes no ordinal assumption.

**Why `drop_first=True`?** k-1 dummies fully represent k categories. The dropped dummy is the reference. Keeping all k creates perfect multicollinearity which destabilises logistic regression (infinite solutions).

In [ ]:
# Ordinal: contract (ordered)
contract_map  = {"Month-To-Month": 0, "One Year": 1, "Two Year": 2}
fe["contract_encoded"] = (
    fe["contract"].astype(str).str.strip().str.title()
    .map(contract_map)
)

unmapped = fe["contract_encoded"].isna().sum()
if unmapped > 0:
    print(f"WARNING: {unmapped} contract values unmapped.")
    print(fe[fe["contract_encoded"].isna()]["contract"].unique())
else:
    print("contract ordinal encoding ok (0=Month-To-Month, 1=One Year, 2=Two Year)")
    print(fe["contract_encoded"].value_counts().sort_index().to_string())

# One-hot: internet_service, payment_method
ohe_cols = [c for c in ["internet_service", "payment_method"] if c in fe.columns]
fe       = pd.get_dummies(fe, columns=ohe_cols, drop_first=True, dtype=int)

new_dummies = [c for c in fe.columns
               if any(c.startswith(p) for p in ["internet_service_", "payment_method_"])]
print(f"\nOne-hot columns created ({len(new_dummies)}): {new_dummies}")
print(f"Shape after encoding: {fe.shape}")

### 9d. Drop columns that must not enter the model

In [ ]:
# Everything here is either an identifier, a target-derived column, an EDA-only bin,
# or a superseded original column. Safe to re-run (only drops columns that exist).
cols_to_drop = [
    "customer_id",                  # identifier
    "total_charges",                # = tenure x monthly_charges -> multicollinear
    "tenure_band",                  # EDA bin -- continuous tenure is more informative
    "contract",                     # superseded by contract_encoded
    "tenure_was_missing",           # cleaning flag -- dropped after EDA showed no signal
    "monthly_charges_was_missing",  # same
    "churn_label",                  # EDA helper column added in Section 8b
]
cols_to_drop = [c for c in cols_to_drop if c in fe.columns]
fe = fe.drop(columns=cols_to_drop)

print(f"Dropped: {cols_to_drop}")
print(f"Shape after dropping: {fe.shape}")

### 9e. Separate X and y

**Why separate now?** Keeping the target in the feature matrix causes data leakage — the model can copy it and achieve 100% training accuracy on meaningless predictions. We separate X and y immediately before scaling and assert the target is not in X.

In [ ]:
y = fe[TARGET].copy().astype(int)
X = fe.drop(columns=[TARGET]).copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Churn rate in y: {y.mean():.1%}")

assert TARGET not in X.columns, "ERROR: target leaked into X!"
print(f"Target not in X: ok")

str_cols = X.select_dtypes(include="object").columns.tolist()
if str_cols:
    print(f"WARNING -- non-numeric columns in X (must encode before modelling): {str_cols}")
else:
    print("All X columns are numeric: ok")

null_X = X.isna().sum().sum()
print(f"Nulls in X: {null_X} {'ok' if null_X == 0 else 'WARNING'}")

print(f"\nAll {X.shape[1]} feature columns:")
print(X.columns.tolist())

### 9f. Scale numeric features

**Why scale?** Linear models and distance-based models treat `tenure` (0-72) and `late_payment_rate` (0-1) as having the same inherent scale — they do not. Without scaling, high-magnitude columns dominate low-magnitude ones regardless of predictive power.

**Why StandardScaler over MinMaxScaler?** MinMaxScaler divides by `(max - min)`. One extreme outlier in `total_logins` or `resolution_time_hours` compresses all other values into a tiny range near zero. StandardScaler centres at 0 with std=1. Outliers remain but all features share comparable variance — more robust for this dataset.

**Why NOT scale binary/ordinal columns?** Scaling a binary column produces ~(-1.3, +0.7) — it adds no model benefit and loses interpretability. Ordinal columns like `contract_encoded` (0, 1, 2) also lose their ordinal meaning when standardised.

**Leakage note:** We fit the scaler on the full X here only to produce the output file. In notebook 04, the scaler is refit inside a pipeline on training data only — fitting on the full dataset before train/test split would leak test-set mean and std into normalisation.

In [ ]:
# Columns to scale: floats OR ints with more than 10 unique values
# This captures continuous + count columns and automatically skips binary/ordinal
cols_to_scale = [
    c for c in X.columns
    if X[c].dtype in ["float64", "float32"]
    or (X[c].dtype in ["int64", "int32"] and X[c].nunique() > 10)
]

print(f"Columns scaled ({len(cols_to_scale)}):")
for c in cols_to_scale:
    print(f"  {c}")

X_scaled                = X.copy()
scaler                  = StandardScaler()
X_scaled[cols_to_scale] = scaler.fit_transform(X[cols_to_scale])

print(f"\nX_scaled shape: {X_scaled.shape}")
print("\nSample check (mean~0, std~1 for scaled cols):")
print(
    X_scaled[cols_to_scale[:5]]
    .describe().T[["mean", "std", "min", "max"]]
    .to_string()
)

---
## 10. Feature Importance Preview

**Why preview before modelling?** It validates the feature engineering. If `num_services`, `risk_score`, or `avg_monthly_spend` appear near the top, they earned their place. If they are at the bottom, they should be reconsidered.

**Why Random Forest?** Fast, no tuning needed, handles all feature types, produces reliable importance ranking. This is a sanity check, not final modelling.

**Why `class_weight='balanced'`?** Without it the forest optimises for the 74% majority class and learns almost nothing about churners.

In [ ]:
# Stratified split so churn rate is preserved in both sets
X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_tr):,} rows  |  churn rate: {y_tr.mean():.1%}")
print(f"Test:  {len(X_te):,} rows  |  churn rate: {y_te.mean():.1%}")

rf_preview = RandomForestClassifier(
    n_estimators=150, max_depth=8,
    class_weight="balanced",
    random_state=42, n_jobs=-1
)
rf_preview.fit(X_tr, y_tr)

importances = (
    pd.Series(rf_preview.feature_importances_, index=X_scaled.columns)
    .sort_values(ascending=False)
)
print("\nTop 20 features by importance:")
print(importances.head(20).to_string())

In [ ]:
top_n        = 20
top_features = importances.head(top_n)
colors       = [C1 if i < 5 else "#888888" for i in range(len(top_features))]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(
    top_features.index[::-1],
    top_features.values[::-1],
    color=colors[::-1], edgecolor="white", linewidth=0.8
)
for i, (val, _name) in enumerate(zip(top_features.values[:5], top_features.index[:5])):
    ax.text(val + 0.001, top_n - 1 - i, f"{val:.4f}", va="center", fontsize=8.5)

ax.set_xlabel("Mean decrease in impurity")
ax.set_title(f"Top {top_n} features — Random Forest preview", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 11. Final Feature Matrix Validation

In [ ]:
print("=" * 55)
print("FEATURE MATRIX -- FINAL VALIDATION")
print("=" * 55)

print(f"X (raw)    : {X.shape[0]:,} x {X.shape[1]}")
print(f"X (scaled) : {X_scaled.shape[0]:,} x {X_scaled.shape[1]}")
print(f"y          : {y.shape[0]:,}")

null_X     = X_scaled.isna().sum().sum()
null_y     = int(y.isna().sum())
str_remain = X_scaled.select_dtypes(include="object").columns.tolist()

print(f"Nulls in X_scaled:    {null_X} {'ok' if null_X == 0 else 'WARNING'}")
print(f"Nulls in y:           {null_y} {'ok' if null_y == 0 else 'WARNING'}")
print(f"Non-numeric in X:     {str_remain if str_remain else 'none ok'}")
print(f"Target not in X:      {'ok' if TARGET not in X.columns else 'LEAKAGE WARNING'}")
print(f"Churn rate:           {y.mean():.1%}  (expected ~26%)")
print(f"\nAll {X.shape[1]} feature columns:")
print(X.columns.tolist())

---
## 12. Save All Outputs

In [ ]:
paths = {
    "X_raw":    f"{PROCESSED_PATH}X_raw.csv",
    "X_scaled": f"{PROCESSED_PATH}X_scaled.csv",
    "y":        f"{PROCESSED_PATH}y.csv",
    "scaler":   f"{PROCESSED_PATH}scaler.pkl",
    "features": f"{PROCESSED_PATH}feature_names.txt",
}

X.to_csv(paths["X_raw"],    index=False)
X_scaled.to_csv(paths["X_scaled"], index=False)
y.to_csv(paths["y"],        index=False, header=True)
joblib.dump(scaler,          paths["scaler"])

with open(paths["features"], "w") as fh:
    fh.write("\n".join(X.columns.tolist()))

print("Saved to data/processed/:")
for name, path in paths.items():
    size_kb = os.path.getsize(path) / 1024
    print(f"  {os.path.basename(path):<25}  {size_kb:>8.1f} KB")

---
## 13. Phase 3 Summary

In [ ]:
print("PHASE 3 COMPLETE -- EDA & FEATURE ENGINEERING")
print("=" * 55)
print()
print("EDA key findings:")
print("  - Churn rate ~26% -- moderate class imbalance")
print("  - Top numeric predictors by Mann-Whitney effect size:")
for _, row in mwu_df.head(5).iterrows():
    print(f"    {row['feature']:<28}  r={row['effect_size_r']:.3f}")
print("  - Top categorical predictors by Cramer's V:")
for _, row in chi2_df.head(5).iterrows():
    print(f"    {row['feature']:<28}  V={row['cramers_v']:.4f}")
print("  - Month-to-Month + new customers = highest churn cell")
print()
print("Feature engineering decisions:")
print("  - 5 new features: avg_monthly_spend, charge_vs_tier_avg,")
print("    logins_per_active_day, num_services, risk_score")
print("  - Yes/No cols mapped 1/0 (No Internet/Phone Service -> 0)")
print("  - contract ordinal: 0=Month-To-Month, 1=One Year, 2=Two Year")
print("  - internet_service + payment_method one-hot (drop_first=True)")
print("  - total_charges dropped (collinear with tenure x monthly_charges)")
print("  - Continuous + count cols scaled with StandardScaler")
print()
print(f"Outputs:")
print(f"  X_raw.csv       {X.shape[0]:,} x {X.shape[1]} (unscaled, for tree models)")
print(f"  X_scaled.csv    {X_scaled.shape[0]:,} x {X_scaled.shape[1]} (scaled, for linear models)")
print(f"  y.csv           {y.shape[0]:,} rows | churn rate {y.mean():.1%}")
print(f"  scaler.pkl      StandardScaler (refit on train-only in nb04)")
print(f"  feature_names   {X.shape[1]} column names")
print()
print("Notebook 04 strategy:")
print("  - Metrics: ROC-AUC + PR-AUC (not accuracy)")
print("  - Imbalance: class_weight=balanced + SMOTE comparison")
print("  - Models: Logistic Regression, Random Forest, XGBoost, LightGBM")
print("  - Tuning: RandomizedSearchCV on best model")
print("  - Interpretation: SHAP values")
print()
print("Next: 04_modelling.ipynb")